**Note:** The original video uses torchtext's legacy `Field` / `TabularDataset` / `Iterator` API, which was removed in torchtext >= 0.12. `torchtext`'s compiled extension (`libtorchtext.so`) is also frequently broken against whatever PyTorch build you have installed (an unresolved ABI-mismatch issue in the now-archived/unmaintained torchtext project), so this notebook avoids importing `torchtext` entirely. Instead it implements the small pieces it actually needs — a `basic_english`-style tokenizer and a `Vocab` class with the same `stoi`/`itos`/`__call__` interface torchtext's `build_vocab_from_iterator` produces — in plain Python, plus a `torch.utils.data.Dataset`/`DataLoader`. The end result (padded integer sequences + labels, batched, on-device) is the same as what the legacy API produced.

In [1]:
import re
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
%matplotlib inline

# torchtext's compiled C++ extension (libtorchtext.so) is frequently ABI-broken
# against whatever PyTorch build is installed, so we avoid importing torchtext
# altogether and implement the tiny bits of it we need (tokenizer + vocab) below
# in plain Python.

In [2]:
# Let's make some fake data!
data = {
    "label": [0, 1, 1],
    "data": [
        "I like eggs and ham.",
        "Eggs I like!",
        "Ham and eggs or just ham?",
    ]
}

is in a dictionary.

In [3]:
df = pd.DataFrame(data)

In [4]:
df.head()

,label,data
0,0,I like eggs and ham.
1,1,Eggs I like!
2,1,Ham and eggs or just ham?


to this dataset so we can see what happens to it in our tokenizer.

In [5]:
df.to_csv('thedata.csv', index=False)

line version of head to see the contents of the file.

In [6]:
!head thedata.csv

label,data
0,I like eggs and ham.
1,Eggs I like!
1,Ham and eggs or just ham?


In [7]:
# TEXT / LABEL "Field" equivalents:
# sequential=True, batch_first=True, lower=True, tokenize='spacy', pad_first=True
# -> a tokenizer function + lowercasing + left-padding done manually below
# LABEL sequential=False, use_vocab=False, is_target=True
# -> labels are just used as plain ints, no vocab needed

def get_tokenizer():
  # comparable to spacy's / torchtext's basic_english word tokenizer,
  # no external download needed
  pattern = re.compile(r"([.!?,;:()\"])")
  def tokenizer(text):
    text = pattern.sub(r" \1 ", text.lower())
    return text.split()
  return tokenizer

tokenizer = get_tokenizer()

csv_df = pd.read_csv('thedata.csv')
csv_df.head()

,label,data
0,0,I like eggs and ham.
1,1,Eggs I like!
2,1,Ham and eggs or just ham?


In [8]:
class Vocab:
  # minimal stand-in for torchtext.vocab.Vocab built via build_vocab_from_iterator
  def __init__(self, tokens_iter, specials=('<unk>', '<pad>')):
    seen = set()
    unique_tokens = []
    for tokens in tokens_iter:
      for tok in tokens:
        if tok not in seen:
          seen.add(tok)
          unique_tokens.append(tok)
    self.itos = list(specials) + sorted(unique_tokens)
    self.stoi = {tok: i for i, tok in enumerate(self.itos)}
    self.default_index = self.stoi[specials[0]]

  def __getitem__(self, token):
    return self.stoi.get(token, self.default_index)

  def __call__(self, tokens):
    return [self[t] for t in tokens]

  def __len__(self):
    return len(self.itos)

  def get_stoi(self):
    return self.stoi

  def get_itos(self):
    return self.itos

def yield_tokens(texts):
  for text in texts:
    yield tokenizer(text)

vocab = Vocab(yield_tokens(csv_df['data']), specials=('<unk>', '<pad>'))

In [9]:
type(vocab)

__main__.Vocab

In [10]:
# equivalent of vocab.stoi
vocab.get_stoi()

{'<unk>': 0,
 '<pad>': 1,
 '!': 2,
 '.': 3,
 '?': 4,
 'and': 5,
 'eggs': 6,
 'ham': 7,
 'i': 8,
 'just': 9,
 'like': 10,
 'or': 11}

In [11]:
# equivalent of vocab.itos
vocab.get_itos()

['<unk>',
 '<pad>',
 '!',
 '.',
 '?',
 'and',
 'eggs',
 'ham',
 'i',
 'just',
 'like',
 'or']

In [12]:
ex_tokens = tokenizer(csv_df['data'].iloc[0])
ex_tokens

['i', 'like', 'eggs', 'and', 'ham', '.']

In [13]:
vocab(ex_tokens)

[8, 10, 6, 5, 7, 3]

In [14]:
class TextDataset(Dataset):
  def __init__(self, df, tokenizer, vocab):
    self.labels = df['label'].values
    self.sequences = [
        torch.tensor(vocab(tokenizer(text)), dtype=torch.long)
        for text in df['data']
    ]

  def __len__(self):
    return len(self.labels)

  def __getitem__(self, idx):
    return self.sequences[idx], self.labels[idx]

dataset = TextDataset(csv_df, tokenizer, vocab)

In [15]:
ex = dataset[0]
ex

(tensor([ 8, 10,  6,  5,  7,  3]), np.int64(0))

In [16]:
type(ex)

tuple

In [17]:
ex[0] # data

tensor([ 8, 10,  6,  5,  7,  3])

In [18]:
ex[1] # label

np.int64(0)

In [19]:
# equivalent of dataset.split(0.66) # default is 0.7
N = len(dataset)
N_train = int(round(N * 0.66))
perm = torch.randperm(N)
train_idx, test_idx = perm[:N_train], perm[N_train:]
train_dataset = torch.utils.data.Subset(dataset, train_idx)
test_dataset = torch.utils.data.Subset(dataset, test_idx)

In [20]:
device = torch.device("mps" if torch.backends.mps.is_available() else ("cuda:0" if torch.cuda.is_available() else "cpu"))
print(device)

mps


In [21]:
pad_idx = vocab['<pad>']

def collate_batch(batch):
  # sort_key=lambda x: len(x.data) equivalent: sort batch by sequence length
  batch = sorted(batch, key=lambda ex: len(ex[0]), reverse=True)
  sequences, labels = zip(*batch)

  # pad_first=True equivalent: pad on the left instead of the right
  reversed_seqs = [seq.flip(0) for seq in sequences]
  padded = pad_sequence(reversed_seqs, batch_first=True, padding_value=pad_idx)
  padded = padded.flip(1)

  targets = torch.tensor(labels, dtype=torch.long)
  return padded.to(device), targets.to(device)

train_iter = DataLoader(train_dataset, batch_size=2, shuffle=True, collate_fn=collate_batch)
test_iter = DataLoader(test_dataset, batch_size=2, shuffle=False, collate_fn=collate_batch)

In [22]:
for inputs, targets in train_iter:
  print("inputs:", inputs, "shape:", inputs.shape)
  print("targets:", targets, "shape:", targets.shape)
  break

inputs: tensor([[ 7,  5,  6, 11,  9,  7,  4],
        [ 1,  8, 10,  6,  5,  7,  3]], device='mps:0') shape: torch.Size([2, 7])
targets: tensor([1, 0], device='mps:0') shape: torch.Size([2])


As you can see the shape of the inputs is 2 by however many tokens are in the longest sentence of the batch.

In [23]:
for inputs, targets in test_iter:
  print("inputs:", inputs, "shape:", inputs.shape)
  print("targets:", targets, "shape:", targets.shape)
  break

inputs: tensor([[ 6,  8, 10,  2]], device='mps:0') shape: torch.Size([1, 4])
targets: tensor([1], device='mps:0') shape: torch.Size([1])


Batches are padded to the longest sequence length possible.

In [24]:
# Exercise: Figure out which sequence of integers goes with which sentence.